In [1]:
%pip install --upgrade pip
#%pip uninstall transformers -y
#%pip uninstall torch torchvision torchaudio -y
#%pip cache purge
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install transformers==4.52.4
%pip install accelerate peft bitsandbytes datasets sentencepiece tokenizers safetensors trl
from transformers import AutoModelForCausalLM, AutoTokenizer

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.
  Using cached transformers-4.52.4-py3-none-any.whl.metadata (38 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.52.4-py3-none-any.whl (10.5 MB)
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:╺━━━━━━━━━━━━━━━━━━━ 1/2 [transformers]
      Successfully uninstalled transformers-4.57.6━━━━━━━━━━━━ 1/2 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transf

In [2]:
import torch
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
train_file = r'./finetune_data/train.jsonl'
valid_file = r'./finetune_data/validation.jsonl'
output_dir = 'peft_output'
num_train_epochs = 3
per_device_train_batch_size = 4
learning_rate = 2e-4
# gpu
if torch.cuda.is_available():    
    use_bf16 = True
    use_fp16= False    
# cpu
else:
    use_bf16 = False
    use_fp16= True

lora_r = 8
lora_alpha = 16
lora_dropout = 0.05
use_4bit = False   
max_leng = 512

LABEL_DESC = {
    'DEF': '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC': '신청/심사/조사/불복/처벌 절차 조항',
    'ORG': '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT': '자격/요건/기준/기간/수치 조건 조항',
    'ETC': '시행일/경과조치/위임 등 기타 조항',
}
print('Config set:', model_name)

Config set: Qwen/Qwen2.5-0.5B-Instruct


In [3]:
# Causal lM : 기본학습 다음토큰을 예측
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

def build_inputs_and_labels(batch, tokenizer, max_length=512):
    '''prompt부분은 loss에서제외(-100) 정답 라벨 토큰만 학습'''
    inputs = []; labels = []
    for p, r in zip(batch['prompt'], batch['response']):
        full = p+' '+ r
        tokenized_full = tokenizer(full, truncation=True, max_length=max_length)
        tokenized_prompt = tokenizer(p, truncation=True, max_length=max_length)
        input_ids =  tokenized_full['input_ids']
        label_ids = input_ids.copy()

        prompt_len = len(tokenized_prompt['input_ids'])
        for i in range(min(prompt_len, len(label_ids))):
            label_ids[i]    = -100         
        inputs.append(input_ids)
        labels.append(label_ids)
    return {'input_ids' : inputs, 'labels':labels}

In [4]:
# 토크나이져, 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
load_kwargs = {'trust_remote_code':True}
if use_4bit:
    load_kwargs.update({'load_in_4bit':True,'device_map':'auto'})
model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
if use_4bit:
    model = prepare_model_for_kbit_training(model)
print('model load')

model load


In [5]:
import torch.nn as nn
def find_lora_target_modules(model):
    linear_leaf_names = set()
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            linear_leaf_names.add(name.split('.')[-1])

    preferred = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    selected = [ m for m in preferred if m in linear_leaf_names]
    if not selected:
        fallback_keywords = ('proj','wq','wk','wv','wo','fc')
        selected = sorted([
            n for n in linear_leaf_names
            if any( k in  n for k in fallback_keywords) and n not in {'lm_head'}
        ])
    if not selected:
        raise ValueError(
            f'LoRA Target Module를 자동 탐색하지 못했습니다. 발견된 leaner leaf name : {linear_leaf_names}'
        )
    return selected, sorted(linear_leaf_names)

target_modules, linear_names = find_lora_target_modules(model)
print('Detected linear left names(samples) :',linear_names[:30])
print('Selected target_modules for LoRA', target_modules)

peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    target_modules=target_modules,
    lora_dropout=lora_dropout,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Detected linear left names(samples) : ['down_proj', 'gate_proj', 'k_proj', 'lm_head', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
Selected target_modules for LoRA ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [6]:
data_files = {'train':train_file, 'validation':valid_file}
dataset = load_dataset('json',data_files=data_files)

def preprocess_batch(batch):
    return build_inputs_and_labels(batch, tokenizer,max_length=max_leng)

tokenized_train = dataset['train'].map(
    lambda x : preprocess_batch(x), batched=True,remove_columns=dataset['train'].column_names
)
tokenized_eval = dataset['validation'].map(
    lambda x : preprocess_batch(x), batched=True,remove_columns=dataset['validation'].column_names
)

def collate_with_padding(features):
    # 가변 길이 샘플을 동적 패딩하고 labels는 -100으로 패딩
    input_ids = [f['input_ids'] for f in features]
    labels = [f['labels'] for f in features]

    padded = tokenizer.pad({'input_ids': input_ids}, padding=True, return_tensors='pt')
    max_len = padded['input_ids'].shape[1]

    padded_labels = []
    for lb in labels:
        padded_labels.append(lb + [-100] * (max_len - len(lb)))

    padded['labels'] = torch.tensor(padded_labels, dtype=torch.long)
    return padded


print('Datasets prepared:', len(tokenized_train), len(tokenized_eval))

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Datasets prepared: 2000 400


In [7]:
from transformers import TrainingArguments, Trainer
base_kwargs = dict(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_train_batch_size,
    num_train_epochs=num_train_epochs,
    learning_rate=learning_rate,
    bf16=use_bf16,
    fp16=use_fp16,
    save_total_limit=3,
    remove_unused_columns=False,
    logging_steps=10,
)

try:
    training_args =  TrainingArguments(**{**base_kwargs, 'evaluation_strategy':'epoch'})
except TypeError:
    training_args =  TrainingArguments(**base_kwargs)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator = collate_with_padding
)
train_result = trainer.train()
print(f"loss : {getattr(train_result,'training_loss', None)}")

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
10,0.933400
20,0.018600
30,0.000400
40,0.000100
50,0.000100
60,0.000100
70,0.000000
80,0.000000
90,0.000000
100,0.000000


loss : 0.012714345072638631


In [9]:
# 학습완료후 PEFT 어뎁터 저장
model.save_pretrained(output_dir)
print('saved PEFT adapters : ',output_dir)

saved PEFT adapters :  peft_output


In [12]:
from peft import PeftModel
reload_kwargs = {'trust_remote_code':True}
if use_4bit:
    reload_kwargs.update({'load_in_4bit' : True, 'device_map':'auto'})
else:
    reload_kwargs.update({
        'torch_dtype':torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    })
base_model_for_eval = AutoModelForCausalLM.from_pretrained(model_name, **reload_kwargs)    
model = PeftModel.from_pretrained(base_model_for_eval,output_dir)
if torch.cuda.is_available() and not use_4bit:
    model.to('cuda')
model.eval()
print('reload PEFT adapter from:',output_dir)
print('Eval model dtype:',next(model.parameters()).dtype)


reload PEFT adapter from: peft_output
Eval model dtype: torch.bfloat16


In [13]:
import random

label_list = list(LABEL_DESC.keys())


def predict_label_from_prompt(prompt_text, max_new_tokens=6):
    model.eval()
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    generated = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    upper = generated.upper()
    for lb in label_list:
        if lb in upper:
            return lb, generated

    first = generated.split()[0].strip(".,:;()[]{}\"'").upper() if generated else ''
    if first in label_list:
        return first, generated
    return 'ETC', generated


val_ds = load_dataset('json', data_files={'validation': valid_file})['validation']
num_eval = min(100, len(val_ds))
indices = random.sample(range(len(val_ds)), k=num_eval)

correct = 0
samples = []
for idx in indices:
    item = val_ds[idx]
    pred, raw = predict_label_from_prompt(item['prompt'])
    gt = item['response']
    correct += int(pred == gt)
    if len(samples) < 5:
        samples.append((gt, pred, raw))

acc = correct / num_eval if num_eval else 0.0
print(f'Validation sample accuracy ({num_eval}개): {acc:.4f}')
print('\n예시 5개 (정답, 예측, 생성원문):')
for row in samples:
    print(row)

Generating validation split: 0 examples [00:00, ? examples/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Validation sample accuracy (100개): 1.0000

예시 5개 (정답, 예측, 생성원문):
('ETC', 'ETC', 'ETC(시행일')
('CRIT', 'CRIT', 'CRIT(자격/요')
('ETC', 'ETC', 'ETC(시행일')
('ORG', 'ORG', 'ORG(기관)')
('DEF', 'DEF', 'DEF(정의/목')


In [15]:
# 단건 추론
test_clause = '신청인은 접수일로부터 14일 이내에 이의신청서를 제출해야 하며, 담당 기관은 30일 내에 심사 결과를 통지한다.'
label_candidates = ', '.join([f"{k}({v})" for k, v in LABEL_DESC.items()])
test_prompt = f"다음 조항을 다음 라벨 중 하나로 분류하시오. 가능한 라벨: {label_candidates}\n조항: {test_clause}\n라벨:"

pred, raw = predict_label_from_prompt(test_prompt)
print('입력 조항:', test_clause)
print('예측 라벨:', pred)
print('생성 원문:', raw)

입력 조항: 신청인은 접수일로부터 14일 이내에 이의신청서를 제출해야 하며, 담당 기관은 30일 내에 심사 결과를 통지한다.
예측 라벨: PROC
생성 원문: PROC(신청/심사
